In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm

import concurrent.futures


import numpy as np
import pandas as pd
import ast
import glob
import pickle
import dask
import os
import itertools

import pickle

#from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from statsmodels.regression.rolling import RollingOLS

from tqdm.notebook import tqdm

from multiprocessing import Pool, cpu_count
from joblib import Parallel, delayed
#from tqdm import tqdm
from collections import Counter
from functools import reduce


import dask
import dask.dataframe as dd
from dask.distributed import Client
from dask.diagnostics import ProgressBar

#client = Client(n_workers=20, memory_limit="10GB", interface='lo')
from concurrent.futures import ThreadPoolExecutor

import dask_ml.cluster as dask_cluster

from pprint import pprint
import os

pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

### Load Augmented DF

In [ ]:
augmented_df = pd.read_parquet("../data/augmented_us_counties.parquet")
augmented_df["date"] = pd.to_datetime(augmented_df["date"])
augmented_df["fips"] = augmented_df["fips"].astype(int)
augmented_df["days_from_start"] = augmented_df["days_from_start"].astype(int)
augmented_df["log_rolled_cases"] = np.log(augmented_df["rolled_cases"] + 1.1)
augmented_df = augmented_df.sort_values(by=["fips","date"])

# Generate 7 day validation data
for shift in range(1,8):
    augmented_df["shifted_log_rolled_cases_{}".format(shift)] = augmented_df.groupby("fips")["log_rolled_cases"].shift(-shift)

# Check for gaps
gt_columns = ["fips", "days_from_start", "date", "log_rolled_cases"] + ["shifted_log_rolled_cases_{}".format(shift) for shift in range(1,8)]
augmented_df_gt = augmented_df[gt_columns]
grouped = augmented_df_gt.groupby('fips')

for fips, group in grouped:
    missing_days = group['days_from_start'].diff().gt(1).sum()
    if missing_days > 0:
        print(f"Gap(s) found in 'days_from_start' for fips {fips}: {missing_days} gap(s)")


#df = augmented_df.copy()
window_sizes = list(range(2,15))
fips_list = augmented_df_gt["fips"].unique()

### Load all $r_{t,c}$

In [ ]:
all_beta_df = dd.read_csv("../benchmark_fixed_window/Fixed_windows_all_beta.csv",assume_missing=True).compute()
all_beta_df["date"] = pd.to_datetime(all_beta_df["date"])
all_beta_df = all_beta_df.sort_values(by=["fips","date"])
(all_beta_df.head())

### Add Predictions

In [ ]:
all_beta_df_results = all_beta_df.copy()
window_sizes = range(2,15)
for window_size in window_sizes:
    for shift in range(1,8):
        all_beta_df_results["predicted_beta_wsize={}_shift={}".format(window_size, shift)] = all_beta_df_results["beta_wsize={}".format(window_size)]*shift + all_beta_df_results["log_rolled_cases"]
all_beta_df_results = all_beta_df_results.drop(columns=["date","log_rolled_cases", "shifted_log_rolled_cases"])
all_beta_df_results = pd.merge(augmented_df_gt, all_beta_df_results, on=["fips", "days_from_start"], how="left")
all_beta_df_results

In [ ]:
# Compute diff = predicted - shifted_log_rolled_cases
import itertools as _it
for window_size in range(2,15):
    for shift in range(1,8):
        pred_col = 'predicted_beta_wsize={}_shift={}'.format(window_size, shift)
        diff_col = 'diff_wsize={}_shift={}'.format(window_size, shift)
        all_beta_df_results[diff_col] = all_beta_df_results[pred_col] - all_beta_df_results['shifted_log_rolled_cases_{}'.format(shift)]


In [ ]:
all_beta_df_results.columns

### Save the Shifts Beta

In [ ]:
wsize_shift = itertools.product(list(range(2,15)), list(range(1,8)))
cols_to_keep = ["fips", "days_from_start", "date"] + ["diff_wsize={}_shift={}".format(window_size, shift) for window_size, shift in wsize_shift]
all_beta_df_results_diff = all_beta_df_results[cols_to_keep]
all_beta_df_results_diff
all_beta_df_results_diff.to_csv("Fixed_Windows_Validation_Diff.csv", index=False)